# Fine-tuning GPT-2 pour l'autocompletion — domaine detection incendie (ISO 7240-14)

Notebook propre et reproductible. Pipeline complet du debut a la fin :

1. Montage Google Drive (sauvegarde systematique, robuste aux sessions qui expirent)
2. Chargement du **corpus nettoye** (`scenarios_fire_detection_clean.txt`)
3. Split **3-way : train / validation / test** (70 / 15 / 15)
4. Fine-tuning GPT-2 adapte a un **petit dataset** :
   - peu d'epochs (early stopping sur la validation)
   - dropout et weight decay renforces
   - petit batch (regularisation implicite)
5. Evaluation : **perplexite** + **Top-1 / Top-3** sur le set de test (jamais vu)

> Methode pour petit dataset : le modele pre-entraine connait deja l'anglais.
> Le fine-tuning doit *adapter* au vocabulaire du domaine, pas reapprendre la langue.
> Donc peu d'epochs + regularisation + early stopping pour eviter le surapprentissage.


## 0. Verifier le GPU

In [ ]:
import torch
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "PAS DE GPU (active-le : Execution > Modifier le type d'execution > GPU)")

GPU : Tesla T4


In [ ]:
!pip install -q transformers datasets accelerate

## 1. Monter Google Drive

On sauvegarde tout sur Drive des le debut. Si la session Colab expire,
rien n'est perdu : le corpus et le modele sont sur le Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/autocomplete'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
print("Projet :", PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Projet : /content/drive/MyDrive/autocomplete


## 2. Charger le corpus nettoye

Depose `scenarios_fire_detection_clean.txt` dans `MyDrive/autocomplete/data/`
(une seule fois). Si le fichier n'y est pas encore, la cellule te propose de l'uploader.

In [ ]:
CLEAN_CORPUS = '/content/drive/MyDrive/scenarios_fire_detection_clean.txt'

if not os.path.exists(CLEAN_CORPUS):
    print("Corpus absent du Drive -> upload manuel")
    from google.colab import files
    up = files.upload()
    fname = list(up.keys())[0]
    with open(CLEAN_CORPUS, 'wb') as f:
        f.write(up[fname])
    print("Copie sur le Drive :", CLEAN_CORPUS)

with open(CLEAN_CORPUS, encoding='utf-8') as f:
    lines = [l.strip() for l in f if l.strip()]
print(f"Corpus charge : {len(lines)} phrases")
print("Exemple :", lines[0])

Corpus charge : 301 phrases
Exemple : Fire detection and alarm systems — Part 14: design, installation, commissioning and service of fire detection and fire alarm systems in and around buildings.


## 3. Split 3-way : train / validation / test (70 / 15 / 15)

- **train** : le modele apprend dessus
- **validation** : sert a l'early stopping (quand arreter pour ne pas surapprendre) — jamais appris
- **test** : jamais vu, sert uniquement a la mesure finale (Top-1/Top-3 + perplexite)

Seed fixe pour la reproductibilite. **Aucune fuite de donnees** : le test est
mis de cote AVANT l'entrainement.

In [ ]:
import random
random.seed(42)
shuffled = lines[:]
random.shuffle(shuffled)

n = len(shuffled)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

train_lines = shuffled[:n_train]
val_lines   = shuffled[n_train:n_train + n_val]
test_lines  = shuffled[n_train + n_val:]

# Sauvegarde des splits (pour que l'evaluation utilise EXACTEMENT le meme test)
for name, data in [('train', train_lines), ('val', val_lines), ('test', test_lines)]:
    with open(f'{PROJECT_DIR}/data/{name}_split.txt', 'w', encoding='utf-8') as f:
        f.write('\n'.join(data))

print(f"Train      : {len(train_lines)} phrases")
print(f"Validation : {len(val_lines)} phrases")
print(f"Test       : {len(test_lines)} phrases")

Train      : 210 phrases
Validation : 45 phrases
Test       : 46 phrases


## 4. Charger GPT-2 avec regularisation renforcee

On augmente le **dropout** (0.2) directement dans la config du modele.
GPT-2 est le modele d'entrainement principal.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, GPT2Config

BASE_MODEL = 'gpt2'
MODEL_DIR  = f'{PROJECT_DIR}/models/gpt2_fire_detection'

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

# Dropout renforce pour petit dataset (defaut 0.1 -> 0.2)
config = GPT2Config.from_pretrained(BASE_MODEL)
config.resid_pdrop  = 0.2
config.embd_pdrop   = 0.2
config.attn_pdrop   = 0.2

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, config=config)
print("GPT-2 charge avec dropout =", config.resid_pdrop)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT-2 charge avec dropout = 0.2


In [ ]:
from datasets import Dataset

def make_ds(lines):
    return Dataset.from_dict({'text': lines})

def tokenize_fn(ex):
    out = tokenizer(ex['text'], truncation=True, padding='max_length', max_length=64)
    out['labels'] = out['input_ids'].copy()
    return out

train_ds = make_ds(train_lines).map(tokenize_fn, batched=True, remove_columns=['text'])
val_ds   = make_ds(val_lines).map(tokenize_fn,   batched=True, remove_columns=['text'])
print("Tokenisation OK")

Map:   0%|          | 0/210 [00:00<?, ? examples/s]

Map:   0%|          | 0/45 [00:00<?, ? examples/s]

Tokenisation OK


## 5. Fine-tuning avec early stopping

Reglages adaptes au petit dataset :
- `num_train_epochs=10` mais **early stopping** arrete bien avant si la validation ne s'ameliore plus
- `weight_decay=0.05` (regularisation)
- `per_device_train_batch_size=2` (petit batch = regularisation implicite)
- evaluation sur la validation a chaque epoch -> on garde le **meilleur** modele (`load_best_model_at_end`)

L'early stopping est la cle : il choisit automatiquement le bon nombre d'epochs.

In [ ]:
from transformers import (DataCollatorForLanguageModeling, Trainer,
                          TrainingArguments, EarlyStoppingCallback)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir=MODEL_DIR,
    num_train_epochs=10,                 # plafond ; early stopping arrete avant
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    warmup_steps=20,
    weight_decay=0.05,                   # regularisation
    lr_scheduler_type='cosine',
    fp16=torch.cuda.is_available(),
    eval_strategy='epoch',               # evalue sur la validation a chaque epoch
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,         # garde le meilleur (selon val loss)
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_steps=10,
    report_to='none',
)

trainer = Trainer(
    model=model, args=args,
    data_collator=collator,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
print("Fine-tuning termine. Meilleur modele sauve sur le Drive :", MODEL_DIR)

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,4.309985,3.456403
2,3.383867,3.188816
3,2.932219,3.116054
4,2.630445,3.106003
5,2.353802,3.100814
6,2.362845,3.117062
7,2.197039,3.120244


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning termine. Meilleur modele sauve sur le Drive : /content/drive/MyDrive/autocomplete/models/gpt2_fire_detection


## 6. Evaluation 1 — Perplexite sur le test

La **perplexite** mesure a quel point le modele predit bien une sequence,
a partir des probabilites qu'il attribue aux bons mots. Plus c'est **bas**, mieux c'est.
On la calcule sur le set de **test** (jamais vu pendant l'entrainement).

On compare le GPT-2 de base vs le GPT-2 fine-tune pour voir le gain du domaine.

In [ ]:
import math

def perplexity(model, lines):
    model.eval()
    device = next(model.parameters()).device
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for line in lines:
            enc = tokenizer(line, return_tensors='pt', truncation=True, max_length=64).to(device)
            out = model(**enc, labels=enc['input_ids'])
            n_tok = enc['input_ids'].size(1)
            total_loss   += out.loss.item() * n_tok
            total_tokens += n_tok
    return math.exp(total_loss / total_tokens)

# Modele de base (reference)
base = AutoModelForCausalLM.from_pretrained('gpt2')
if torch.cuda.is_available(): base = base.cuda()

ppl_base = perplexity(base, test_lines)
ppl_ft   = perplexity(model, test_lines)

print(f"Perplexite GPT-2 de base      : {ppl_base:.2f}")
print(f"Perplexite GPT-2 fine-tune    : {ppl_ft:.2f}")
print(f"Amelioration                  : {(ppl_base - ppl_ft) / ppl_base * 100:.1f} %")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Perplexite GPT-2 de base      : 71.14
Perplexite GPT-2 fine-tune    : 19.59
Amelioration                  : 72.5 %


## 7. Evaluation 2 — Top-1 / Top-3 sur le test

Protocole identique a celui decrit a Gabriela : pour chaque mot, on donne au modele
le contexte + les **2 premieres lettres**, et on verifie s'il predit le bon mot.
- **Top-1** : le bon mot est la 1re proposition
- **Top-3** : le bon mot est dans les 3 premieres

In [ ]:
import re
import torch

def tokenize_words(text):
    return re.findall(r"\b[a-zA-Z]+\b", text.lower())

# --- Pre-calcul : pour chaque token du vocabulaire, son 1er caractere alphabetique ---
# Permet de filtrer par prefixe directement sur les logits (rapide et correct).
vocab_size = len(tokenizer)
tok_first_alpha = []
for tid in range(vocab_size):
    s = tokenizer.decode([tid])
    s = s.strip().lower()
    m = re.match(r"[a-z]", s)
    tok_first_alpha.append(s if m else "")

def predict_topk(model, context_words, prefix, k=3):
    """Renvoie les k mots les plus probables qui commencent par `prefix`,
    en lisant directement les logits du modele (pas de generation)."""
    device = next(model.parameters()).device
    sentence = " ".join(context_words).strip() or tokenizer.eos_token
    enc = tokenizer(sentence, return_tensors="pt").to(device)

    with torch.no_grad():
        logits = model(**enc).logits[0, -1, :]   # logits du prochain token

    # Classe tous les tokens par probabilite decroissante
    order = torch.argsort(logits, descending=True).tolist()

    preds, seen = [], set()
    for tid in order:
        w = tok_first_alpha[tid]
        if not w:
            continue
        w = re.sub(r"[^a-z]", "", w)
        if not w or w in seen:
            continue
        if prefix and not w.startswith(prefix):
            continue
        preds.append(w)
        seen.add(w)
        if len(preds) >= k:
            break
    return preds

def eval_topk(model, lines, prefix_len=2):
    top1, top3, total = 0, 0, 0
    for line in lines:
        words = tokenize_words(line)
        for i in range(1, len(words)):
            target = words[i]
            if len(target) <= prefix_len:
                continue
            context = words[:i]
            prefix  = target[:prefix_len]
            preds = predict_topk(model, context, prefix, k=3)
            if preds and preds[0] == target: top1 += 1
            if target in preds:              top3 += 1
            total += 1
    return {"Top-1": round(top1/total*100, 1) if total else 0,
            "Top-3": round(top3/total*100, 1) if total else 0,
            "tests": total}

print("Evaluation Top-k (logits directs) sur le test...")
res_ft = eval_topk(model, test_lines)
print("GPT-2 fine-tune :", res_ft)

Evaluation Top-k (logits directs) sur le test...
GPT-2 fine-tune : {'Top-1': 71.2, 'Top-3': 84.8, 'tests': 671}


In [ ]:
# --- Comparaison : GPT-2 de base (non fine-tune) sur le MEME test ---
from transformers import AutoModelForCausalLM

base = AutoModelForCausalLM.from_pretrained("gpt2")
if torch.cuda.is_available():
    base = base.cuda()

print("Evaluation Top-k du GPT-2 de base...")
res_base = eval_topk(base, test_lines)

print()
print("=== Comparaison Top-k (meme test) ===")
print("GPT-2 de base   :", res_base)
print("GPT-2 fine-tune :", res_ft)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluation Top-k du GPT-2 de base...

=== Comparaison Top-k (meme test) ===
GPT-2 de base   : {'Top-1': 58.4, 'Top-3': 73.9, 'tests': 671}
GPT-2 fine-tune : {'Top-1': 71.2, 'Top-3': 84.8, 'tests': 671}


In [ ]:
def next_words(sentence, k=5):
    """Donne les k mots les plus probables apres `sentence`, avec ton modele fine-tune."""
    device = next(model.parameters()).device
    enc = tokenizer(sentence, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**enc).logits[0, -1, :]
    order = torch.argsort(logits, descending=True).tolist()

    preds, seen = [], set()
    for tid in order:
        w = tokenizer.decode([tid]).strip().lower()
        w = re.sub(r"[^a-z]", "", w)
        if w and w not in seen:
            preds.append(w); seen.add(w)
        if len(preds) >= k:
            break
    return preds

# --- Tests : tape une phrase, vois les mots suivants proposes ---
for phrase in [
    "The fire detection",
    "A manual call",
    "Smoke detectors shall be",
    "Each detection",
]:
    print(f"{phrase!r:40} ->  {next_words(phrase)}")

'The fire detection'                     ->  ['and', 'system', 'zone', 'equipment', 'systems']
'A manual call'                          ->  ['point', 'centre', 'for', 'to', 'indicator']
'Smoke detectors shall be'               ->  ['installed', 'located', 'provided', 'mounted', 'used']
'Each detection'                         ->  ['zone', 'zones', 'and', 'range', 'point']


In [ ]:
phrase = ""
while True:
    mot = input("mot suivant (vide pour arreter) : ")
    if not mot.strip():
        break
    phrase = (phrase + " " + mot).strip()
    print(f"  phrase : {phrase}")
    print(f"  suggestions -> {next_words(phrase)}")

mot suivant (vide pour arreter) : the
  phrase : the
  suggestions -> ['the', 'of', 'a', 'design', 'line']
mot suivant (vide pour arreter) : fire
  phrase : the fire
  suggestions -> ['alarm', 'detection', 'protection', 'control', 'extingu']
mot suivant (vide pour arreter) : detection
  phrase : the fire detection
  suggestions -> ['and', 'system', 'zone', 'equipment', 'systems']
mot suivant (vide pour arreter) : system
  phrase : the fire detection system
  suggestions -> ['shall', 'is', 'may', 'installed', 'for']
mot suivant (vide pour arreter) : 
